# Spam Email Detector — Machine Learning Project

**Project**: Building a spam classifier using machine learning techniques
**Tools**: Python, scikit-learn, pandas, matplotlib, seaborn
**Dataset**: SMS Spam Collection (UCI Machine Learning Repository)
**Platform**: Google Colab

---

## Project Overview

Spam detection is a classic **text classification** problem in machine learning. Every day, billions of emails and SMS messages are filtered automatically — the techniques in this notebook are the same ones used (in simplified form) by services like Gmail and Outlook.

### What this notebook does

1. **Load** a labeled dataset of spam and legitimate (ham) messages
2. **Explore** the data with statistics and visualizations
3. **Preprocess** the text (cleaning, normalization)
4. **Vectorize** text into numerical features using **TF-IDF**
5. **Train** four supervised learning algorithms
6. **Evaluate** them using accuracy, precision, recall, F1-score, and confusion matrices
7. **Test** the best model on custom messages
8. **Save** the trained model for future use


## 1. Setup — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
print("All libraries imported successfully")


## 2. Load the Dataset

We use the **SMS Spam Collection** dataset from the UCI Machine Learning Repository — a public dataset of 5,574 SMS messages labeled as either *spam* or *ham* (legitimate).

The cell below downloads the dataset directly so the notebook is fully self-contained.


In [ ]:
import urllib.request
import zipfile
import os

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
urllib.request.urlretrieve(url, "spam.zip")

with zipfile.ZipFile("spam.zip", "r") as z:
    z.extractall("spam_data")

df = pd.read_csv(
    "spam_data/SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "text"]
)

print(f"Dataset loaded — {df.shape[0]} messages, {df.shape[1]} columns")
df.head()


In [ ]:
# Quick info about the dataset
print("Dataset Info:")
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())


## 3. Exploratory Data Analysis

Before modeling we explore the data to understand:
- How balanced are the spam vs ham classes?
- Are spam messages longer or shorter than legitimate ones?
- What words are most common in each class?


In [ ]:
# Class distribution
print("Class Distribution:")
print(df['label'].value_counts())
print(f"\nSpam ratio: {(df['label']=='spam').mean()*100:.2f}%")
print(f"Ham ratio : {(df['label']=='ham').mean()*100:.2f}%")

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
df['label'].value_counts().plot(kind='bar', ax=ax[0], color=['#2ecc71', '#e74c3c'])
ax[0].set_title('Class Distribution (Counts)', fontsize=13)
ax[0].set_ylabel('Number of Messages')
ax[0].tick_params(axis='x', rotation=0)

df['label'].value_counts().plot(kind='pie', ax=ax[1], autopct='%1.1f%%',
                                 colors=['#2ecc71', '#e74c3c'], startangle=90)
ax[1].set_title('Class Distribution (Percentage)', fontsize=13)
ax[1].set_ylabel('')
plt.tight_layout()
plt.show()


In [ ]:
# Message length analysis
df['length'] = df['text'].apply(len)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
df[df['label']=='ham']['length'].hist(bins=50, ax=ax[0], color='#2ecc71', alpha=0.7)
ax[0].set_title('Message Length — HAM', fontsize=13)
ax[0].set_xlabel('Length (characters)')
ax[0].set_ylabel('Frequency')

df[df['label']=='spam']['length'].hist(bins=50, ax=ax[1], color='#e74c3c', alpha=0.7)
ax[1].set_title('Message Length — SPAM', fontsize=13)
ax[1].set_xlabel('Length (characters)')
ax[1].set_ylabel('Frequency')
plt.tight_layout()
plt.show()

print("\nLength statistics by class:")
print(df.groupby('label')['length'].describe().round(2))


In [ ]:
# Most common words in each class
from collections import Counter

def get_top_words(texts, n=15):
    words = ' '.join(texts).lower().split()
    words = [w.strip(string.punctuation) for w in words if len(w) > 3]
    return Counter(words).most_common(n)

ham_words = get_top_words(df[df['label']=='ham']['text'])
spam_words = get_top_words(df[df['label']=='spam']['text'])

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

ham_w, ham_c = zip(*ham_words)
ax[0].barh(ham_w, ham_c, color='#2ecc71')
ax[0].set_title('Top 15 Words in HAM Messages', fontsize=13)
ax[0].invert_yaxis()

spam_w, spam_c = zip(*spam_words)
ax[1].barh(spam_w, spam_c, color='#e74c3c')
ax[1].set_title('Top 15 Words in SPAM Messages', fontsize=13)
ax[1].invert_yaxis()

plt.tight_layout()
plt.show()


## 4. Text Preprocessing

Raw text needs to be cleaned before machine learning algorithms can use it. Our pipeline:

1. **Lowercase** — "FREE" and "free" should be treated the same
2. **Remove URLs** — links don't add semantic meaning
3. **Remove numbers** — phone numbers, prices etc. add noise
4. **Remove punctuation** — keep only alphabetic words
5. **Collapse whitespace** — single spaces only


In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned'] = df['text'].apply(preprocess_text)

print("Examples — BEFORE vs AFTER preprocessing:\n")
for i in [0, 2, 5, 8]:
    print(f"Original : {df['text'].iloc[i]}")
    print(f"Cleaned  : {df['cleaned'].iloc[i]}")
    print(f"Label    : {df['label'].iloc[i]}")
    print("-" * 80)


In [ ]:
# Encode labels: ham=0, spam=1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})
df[['label', 'label_num', 'text', 'cleaned']].head()


## 5. Train/Test Split & TF-IDF Vectorization

We split the data **80/20** (training/testing) and convert text into numerical vectors using **TF-IDF** (Term Frequency–Inverse Document Frequency). TF-IDF gives higher weight to words that appear in a specific message but are rare across the whole corpus — these are the words that carry meaning.

We use:
- `ngram_range=(1, 2)` — both single words and word pairs (e.g. "free entry")
- `stop_words='english'` — remove common words like "the", "is", "and"
- `min_df=2` — ignore words appearing in only one message
- `max_df=0.95` — ignore words appearing in 95%+ of messages


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned'],
    df['label_num'],
    test_size=0.2,
    random_state=42,
    stratify=df['label_num']
)

print(f"Training set: {len(X_train)} messages")
print(f"Test set    : {len(X_test)} messages")
print(f"\nClass balance in train: {y_train.value_counts(normalize=True).round(3).to_dict()}")
print(f"Class balance in test : {y_test.value_counts(normalize=True).round(3).to_dict()}")


In [ ]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"TF-IDF feature matrix shape: {X_train_vec.shape}")
print(f"Number of features (vocabulary size): {len(vectorizer.get_feature_names_out())}")
print(f"\nSample features: {list(vectorizer.get_feature_names_out()[100:110])}")


## 6. Train Multiple Models

We train **four different classifiers** to see which works best:

| Model | Why use it |
|---|---|
| **Multinomial Naive Bayes** | Classic baseline for text — fast, works well with word counts |
| **Logistic Regression** | Strong linear baseline, interpretable coefficients |
| **Linear SVM** | Often state-of-the-art for high-dimensional text classification |
| **Random Forest** | Non-linear ensemble — useful for comparison |


In [ ]:
models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Linear SVM': LinearSVC(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

results = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_test_vec)
    results[name] = {
        'model': model,
        'predictions': preds,
        'accuracy': accuracy_score(y_test, preds),
        'precision': precision_score(y_test, preds),
        'recall': recall_score(y_test, preds),
        'f1': f1_score(y_test, preds)
    }
    print(f"   Done — Accuracy: {results[name]['accuracy']:.4f}, F1: {results[name]['f1']:.4f}")

print("\nAll models trained successfully")


## 7. Model Evaluation

We compare all four models across multiple metrics. For spam detection:

- **Accuracy** — overall correctness
- **Precision** — of messages flagged as spam, how many were actually spam (avoids false alarms on real mail)
- **Recall** — of all real spam, how many did we catch (don't miss spam)
- **F1-score** — harmonic mean — best single metric when both matter


In [ ]:
results_df = pd.DataFrame({
    name: [r['accuracy'], r['precision'], r['recall'], r['f1']]
    for name, r in results.items()
}, index=['Accuracy', 'Precision', 'Recall', 'F1-Score']).T

print("Model Performance Comparison:\n")
print(results_df.round(4))

ax = results_df.plot(kind='bar', figsize=(12, 6), colormap='viridis')
ax.set_title('Model Performance Comparison', fontsize=14)
ax.set_ylabel('Score')
ax.set_xticklabels(results_df.index, rotation=15)
ax.legend(loc='lower right')
ax.set_ylim(0.85, 1.0)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)
plt.tight_layout()
plt.show()


In [ ]:
# Confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (name, r) in zip(axes.flatten(), results.items()):
    cm = confusion_matrix(y_test, r['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'],
                cbar=False, annot_kws={'size': 14})
    ax.set_title(f'{name}\nAccuracy: {r["accuracy"]:.4f} | F1: {r["f1"]:.4f}', fontsize=12)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — All Models', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Identify best model by F1-score
best_name = max(results, key=lambda k: results[k]['f1'])
best_model = results[best_name]['model']

print(f"Best Model: {best_name}")
print(f"   Accuracy : {results[best_name]['accuracy']:.4f}")
print(f"   Precision: {results[best_name]['precision']:.4f}")
print(f"   Recall   : {results[best_name]['recall']:.4f}")
print(f"   F1-Score : {results[best_name]['f1']:.4f}")
print("\nDetailed classification report:\n")
print(classification_report(
    y_test,
    results[best_name]['predictions'],
    target_names=['Ham', 'Spam']
))


## 8. Test on Custom Messages

The real test — let's try our best model on **completely new messages** it has never seen before.


In [ ]:
custom_messages = [
    "Hey, are we still meeting for lunch tomorrow at 1pm?",
    "CONGRATULATIONS! You've won a $1000 Walmart gift card. Click here to claim now!",
    "Mom, I'll be home late tonight, don't wait for dinner.",
    "URGENT: Your account has been compromised. Verify your password at this link immediately.",
    "Can you send me the notes from today's lecture?",
    "FREE entry in our weekly competition! Text WIN to 80086 to enter. T&Cs apply.",
    "Meeting moved to 3pm in conference room B.",
    "You have been SELECTED for a $500 cash prize! Reply YES to claim now!!!",
    "Don't forget to submit your assignment before midnight.",
    "Click this link to get FREE iPhone 15. Limited offer, hurry up!"
]

custom_clean = [preprocess_text(m) for m in custom_messages]
custom_vec = vectorizer.transform(custom_clean)
predictions = best_model.predict(custom_vec)

print(f"Predictions using {best_name}:\n")
print("=" * 80)
for msg, pred in zip(custom_messages, predictions):
    label = "SPAM" if pred == 1 else "HAM "
    icon = "[X]" if pred == 1 else "[OK]"
    print(f"{icon} [{label}] {msg}")
    print("-" * 80)


In [ ]:
# Interactive testing — try your own message
def predict_message(message):
    cleaned = preprocess_text(message)
    vec = vectorizer.transform([cleaned])
    pred = best_model.predict(vec)[0]
    label = "SPAM" if pred == 1 else "HAM"
    print(f"Message: {message}")
    print(f"Cleaned: {cleaned}")
    print(f"Prediction: {label}")
    return label

# Try changing this to test your own messages
predict_message("Win a free iPhone now! Click here to claim your prize!")
print()
predict_message("Hey can you call me back when you get a chance")


## 9. Save the Trained Model

We save the model and vectorizer so they can be reused without retraining. In a real application this would be loaded by a web server to classify incoming messages in real time.


In [ ]:
import joblib

joblib.dump(best_model, 'spam_classifier.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

print(f"Model ({best_name}) saved as: spam_classifier.pkl")
print(f"Vectorizer saved as: tfidf_vectorizer.pkl")
print("\nTo reload later:")
print("  model = joblib.load('spam_classifier.pkl')")
print("  vectorizer = joblib.load('tfidf_vectorizer.pkl')")


## 10. Conclusion

### Summary

In this project we built an end-to-end spam detection system:

- Loaded a public dataset of **5,574 labeled messages** from UCI
- Performed exploratory analysis (class balance, length distribution, top words)
- Cleaned text using a custom preprocessing pipeline
- Converted text into numerical features using **TF-IDF with bigrams**
- Trained and compared **four ML algorithms** (Naive Bayes, Logistic Regression, Linear SVM, Random Forest)
- Achieved **>97% accuracy** on the test set
- Validated the model on completely new custom messages

### Key Learnings

- **TF-IDF** is highly effective for text classification — no deep learning needed at this data scale
- **Linear models** (Naive Bayes, Logistic Regression, Linear SVM) are very competitive for high-dimensional sparse text data and far faster than ensembles
- **Mild class imbalance** (~13% spam) is handled well without resampling
- **Preprocessing matters** — even simple cleaning (lowercasing, removing punctuation/numbers) significantly affects results

### Possible Extensions

- Use **word embeddings** (Word2Vec, GloVe) or **transformer models** (BERT, DistilBERT) for richer representations
- Apply **SMOTE** or class weighting for stronger imbalance handling
- Try **character-level n-grams** to catch obfuscated spam ("fr33 m0n3y")
- Test on **email datasets** (Enron) with longer texts and more diverse content
- Deploy as a **REST API** using Flask or FastAPI for real-time predictions
- Build a simple **web interface** with Streamlit to demo the model

---

**End of Project**
